# PGA (Programable Gain Amplifier) - Sizing & Eqution Derivations

**Block:** 'pgaaaf' integrator OTA (NanoVolt, GF180MCU 'gf180mcuD', 3.3 V devices)
**Reference:** C. Zhang, L. Shang, Y. Wang, and L. Tang, “A CMOS Programmable Fourth-Order Butterworth Active-RC Low-Pass Filter,” Electronics, vol. 9, no. 2, Art. no. 204, 2020.
**Companion data:** `../../../Sizing/techsweep_gf180mcu_*` — the same gm/ID tables used by `techsweep_gf180mcu_plots.ipynb`.

This notebook does two things:

1. **Derives every equation** used to size the OTA — from the *filter-level* budgets (GBW, load, phase margin, slew rate, power) down to the *two-stage Miller topology* small-signal transfer function (dominant pole, non-dominant pole, RHP zero, nulling resistor), step by step.
2. **Turns those equations into GF180 numbers** by reading `gm/ID`, `Jd = ID/W`, `VT` and `STH` (noise) from the technology sweep, then prints a recommended sizing table.

## 0. Specifications (from the PGA spec sheet, propagated to the OTA)

| Parameter | Value | Drives |
|---|---|---|
| Topology | Tow-Thomas biquad, fully diff. | 2 OTAs total (no separate inverter needed) |
| Filter order / shape | 2nd-order Butterworth, Q = 0.7071 | sets integrator loading (informational here) |
| −3 dB cutoff f0 | 4.0 MHz | slew-rate / large-signal check |
| OTA GBW | > 40 MHz (10× cutoff) | dominant-pole / Cc sizing |
| Load per OTA, CL | C_int + parasitic ≈ 2.5 pF | Cint = 2 pF integrator cap (from `pga_towthomas_sizing.ipynb`) + ~0.5 pF estimate |
| Output swing | 2.0 Vpp differential (1.0 Vpp single-ended) | headroom check |
| Total PGA power | < 400 µW | current budget, split across 2 OTAs |
| Supply VDDA | 3.3 V (GF180MCU 3.3 V devices) | all Vov / gm-ID lookups |

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import constants as sc
import gzip, shutil, tempfile
try:
    from pygmid import Lookup as lk          # gm/ID table interpolation
except ImportError as e:
    raise ImportError("pygmid missing -> in the IIC-OSIC-TOOLS container: pip install pygmid") from e

# ---- physical constants (scipy CODATA) ----
k  = sc.Boltzmann          # Boltzmann [J/K]
q  = sc.elementary_charge  # electron charge [C]
T  = 300.0                 # temperature [K] (~27 C)
kT = k*T

# ---- filter / PGA specs ----
VDDA      = 3.3            # V, GF180MCU 3.3V devices (matches the Sizing/ tables)
f0        = 4.0e6          # Hz, -3 dB cutoff (Butterworth biquad)
Q_bw      = 1/np.sqrt(2)
Cint      = 2.0e-12        # F, integrator cap from pga_towthomas_sizing.ipynb
Cpar_est  = 0.5e-12        # F, output/routing parasitic estimate
CL        = Cint + Cpar_est

GBW_spec  = 40e6           # Hz, ">10x cutoff" spec floor
GBW_margin= 1.25           # design margin over the spec floor
GBW       = GBW_margin*GBW_spec

PM_target_deg = 65.0       # phase margin target (60 deg is the bare minimum; design for extra margin)
k_pm      = 2.5            # p2 >= k_pm * GBW rule of thumb for ~65 deg PM w/ Miller comp
                            # (Allen&Holberg / Razavi: k_pm~2.2 for 60 deg, ~3 for 70 deg)

n_ota     = 2               # both Tow-Thomas integrators use this OTA (fully-diff needs no separate inverter)
P_total_budget = 400e-6     # W, whole PGA
P_ota_budget   = P_total_budget/n_ota    # W, per-OTA share (bias/CMFB overhead handled in Section E)

Vswing_diff_pp = 2.0        # V, differential full-scale
Vswing_se_pp   = Vswing_diff_pp/2   # V, single-ended equivalent swing

print(f"CL (per OTA)      = {CL*1e12:.2f} pF")
print(f"GBW design target = {GBW/1e6:.1f} MHz (spec floor {GBW_spec/1e6:.0f} MHz x {GBW_margin:.2f} margin)")
print(f"P budget / OTA    = {P_ota_budget*1e6:.1f} uW  ({n_ota} OTAs share {P_total_budget*1e6:.0f} uW total)")
print(f"Single-ended swing target = {Vswing_se_pp:.2f} Vpp")


CL (per OTA)      = 2.50 pF
GBW design target = 50.0 MHz (spec floor 40 MHz x 1.25 margin)
P budget / OTA    = 200.0 uW  (2 OTAs share 400 uW total)
Single-ended swing target = 1.00 Vpp


## A. From PGA/filter specs to OTA budgets

### A.1 Dominant-pole / Miller-cap sizing
For a two-stage Miller-compensated op-amp, the loop's unity-gain frequency (≈ GBW, since the dominant pole plus one-pole rolloff crosses 0 dB there) is set by the **first-stage transconductance** and the **compensation capacitor**:
$$\mathrm{GBW}\approx\frac{g_{m1}}{2\pi C_c}\quad\Rightarrow\quad \boxed{\,C_c=\dfrac{g_{m1}}{2\pi\,\mathrm{GBW}}\,}.$$
$C_c$ cancels out of the closed-loop DC gain (that only needs $g_{m1}R_{o1}g_{m6}R_{o2}\gg1$); it only sets *where* the loop crosses unity gain.

### A.2 Phase margin → non-dominant pole → required $g_{m6}$
The output pole $p_2\approx g_{m6}/C_L$ must sit well past GBW. Using the standard two-pole approximation $\mathrm{PM}\approx90^\circ-\arctan(\mathrm{GBW}/p_2)$, a target PM translates into a minimum pole-separation ratio $k_\mathrm{pm}=p_2/\mathrm{GBW}$ (with the correction for the feedforward RHP zero folded into the commonly tabulated $k_\mathrm{pm}\approx2.2\text{–}3$ for 60°–70° PM, since the zero eats phase that the simple arctan expression alone doesn't capture — Allen & Holberg / Razavi):
$$p_2=\frac{g_{m6}}{C_L}\ge k_\mathrm{pm}\cdot\mathrm{GBW}\quad\Rightarrow\quad \boxed{\,g_{m6}\ge k_\mathrm{pm}\,\mathrm{GBW}\,C_L\,}.$$
This is a **fixed target number** — independent of how the power budget is split between stages — so it is computed first and then used to size stage 2's bias current directly (Section D).

### A.3 Slew-rate requirement (large-signal, full-scale sine at f0)
The compensation cap is charged/discharged by the stage-1 tail current during slewing, $\mathrm{SR}=I_{5}/C_c$. For **no slew-induced distortion** on a full-scale sine at the cutoff frequency, the OTA's linear SR must beat the sine's maximum slope:
$$\boxed{\,\mathrm{SR}_\mathrm{req}=2\pi f_0\,\hat V_\mathrm{out,se}\,},\qquad \hat V_\mathrm{out,se}=\frac{V_\mathrm{swing,se,pp}}{2}.$$

### A.4 Power budget → total tail current
With the whole PGA power ceiling split across the `n_ota` OTAs in the signal path:
$$\boxed{\,I_\mathrm{total}=\dfrac{P_\mathrm{OTA\,budget}}{V_\mathrm{DDA}}\,}=I_5+I_7,$$
where $I_5$ is the input-pair (stage 1) tail current and $I_7$ is the stage-2 bias current. Because $g_{m6}$ is already pinned by A.2, $I_7=g_{m6}/(g_m/I_D)_\mathrm{stage2}$ is fixed once a stage-2 inversion level is chosen, and whatever remains of $I_\mathrm{total}$ is left for $I_5$ — this ordering (constraint-driven, not a free split) is used in Section D.


In [4]:

# A.1-A.3 : purely algebraic budgets (device-independent)
Vout_pk_se = Vswing_se_pp/2
SR_req = 2*np.pi*f0*Vout_pk_se

print(f"A.1  GBW design       = {GBW/1e6:.1f} MHz")
print(f"A.2  k_pm (for PM~{PM_target_deg:.0f} deg) = {k_pm:.2f}  ->  gm6_target = k_pm*GBW*CL")
gm6_target = k_pm*GBW*CL
print(f"     gm6_target        = {gm6_target*1e6:.1f} uS")
print(f"A.3  SR required       = {SR_req/1e6:.2f} V/us   (full-scale {Vswing_se_pp:.2f}Vpp sine @ f0={f0/1e6:.1f}MHz)")
print(f"A.4  I_total budget/OTA= {P_ota_budget/VDDA*1e6:.2f} uA  (P={P_ota_budget*1e6:.0f}uW @ VDDA={VDDA}V)")


A.1  GBW design       = 50.0 MHz
A.2  k_pm (for PM~65 deg) = 2.50  ->  gm6_target = k_pm*GBW*CL
     gm6_target        = 312.5 uS
A.3  SR required       = 12.57 V/us   (full-scale 1.00Vpp sine @ f0=4.0MHz)
A.4  I_total budget/OTA= 60.61 uA  (P=200uW @ VDDA=3.3V)


## B. Two-stage Miller OTA equations, derived

**Topology** (single-ended half-circuit of the fully-differential OTA): stage 1 is a differential pair M1/M2 (tail $I_5$) with a current-mirror load M3/M4, output node **X**; stage 2 is a common-source M6 with current-source load M7, output node **Out** (loaded by $C_L$). A Miller cap $C_c$ in series with a nulling resistor $R_z$ connects **Out** back to **X**.

### B.1 Small-signal nodal equations
Let $R_{o1}=r_{o2}\|r_{o4}$ (node X) and $R_{o2}=r_{o6}\|r_{o7}$ (node Out), and let $Z_c(s)=R_z+\frac{1}{sC_c}$ be the compensation branch impedance. KCL at the two nodes:

$$g_{m1}V_\mathrm{in}+\frac{V_X}{R_{o1}}+\frac{V_X-V_\mathrm{out}}{Z_c}=0,\qquad
g_{m6}V_X+\frac{V_\mathrm{out}}{R_{o2}}+sC_LV_\mathrm{out}+\frac{V_\mathrm{out}-V_X}{Z_c}=0.$$

Solving this pair symbolically for $H(s)=V_\mathrm{out}/V_\mathrm{in}$ (done in code below with SymPy) gives the standard two-pole-one-zero op-amp transfer function. Two results matter for sizing:


In [5]:

import sympy as sp
sp.init_printing()

s, gm1s, gm6s, Ro1, Ro2, CLs, Ccs, Rzs, Vin = sp.symbols('s g_m1 g_m6 R_o1 R_o2 C_L C_c R_z V_in', positive=True)
VX, Vout = sp.symbols('V_X V_out')

Zc = Rzs + 1/(s*Ccs)

eqX   = sp.Eq(gm1s*Vin + VX/Ro1 + (VX-Vout)/Zc, 0)
eqOut = sp.Eq(gm6s*VX  + Vout/Ro2 + s*CLs*Vout + (Vout-VX)/Zc, 0)

sol = sp.solve([eqX, eqOut], [VX, Vout], dict=True)[0]
H = sp.simplify(sol[Vout]/Vin)
H_num, H_den = sp.fraction(sp.together(H))
H_num = sp.expand(H_num)
H_den = sp.expand(H_den)
print("Numerator (factor for the zero):")
sp.factor(H_num)


Numerator (factor for the zero):


Under the usual $R_{o1},R_{o2}\to\infty$ (dominant-pole) approximation, the closed-form poles/zero collapse to the textbook results:

$$\boxed{\;p_1\approx-\dfrac{1}{R_{o1}\,C_c\,g_{m6}R_{o2}}\;}\ \text{(dominant, Miller-multiplied)},\qquad
\boxed{\;p_2\approx-\dfrac{g_{m6}}{C_L}\;}\ \text{(non-dominant, output pole)},$$

$$\boxed{\;z=\dfrac{1}{C_c\!\left(\dfrac{1}{g_{m6}}-R_z\right)}\;}.$$

$R_z=0$ gives the classic **RHP** zero at $g_{m6}/C_c$ (eats phase margin). Two useful choices of $R_z$:

* $R_z=\dfrac{1}{g_{m6}}$ pushes the zero to $\pm\infty$ (removes it from the transfer function);
* $R_z=\dfrac{C_c+C_L}{g_{m6}C_c}$ places the zero exactly on top of $-p_2$, **cancelling** the non-dominant pole and buying extra phase margin for the same $C_c$ — this is the choice used below.

### B.2 Design equations as functions


In [6]:

def cc_from_gbw(gm1, GBW_hz):
    "A.1: Cc = gm1/(2*pi*GBW)"
    return gm1/(2*np.pi*GBW_hz)

def gm6_from_pm(k_pm, GBW_hz, CL_f):
    "A.2: gm6 >= k_pm * GBW * CL"
    return k_pm*GBW_hz*CL_f

def rz_zero_cancel(Cc, CL_f, gm6):
    "B.1: Rz that places the LHP zero on top of -p2 (pole-zero cancellation)"
    return (Cc+CL_f)/(gm6*Cc)

def slew_rate(I5, Cc):
    "A.3: SR = I5/Cc (Cc-limited slewing, stage 1)"
    return I5/Cc

def dc_gain(gm1, Ro1, gm6, Ro2):
    return gm1*Ro1*gm6*Ro2

def input_ref_noise_psd(gm1, gm3, gamma=2/3):
    "single-ended stage-1-dominated thermal noise PSD, referred to input [V^2/Hz]"
    return (16*kT/3)*(1/gm1)*(1+gm3/gm1)

def integrated_noise(psd_v2hz, noise_bw_hz):
    "PSD * brick-wall-equivalent noise bandwidth -> integrated noise variance [V^2]"
    return psd_v2hz*noise_bw_hz

print("equation helpers defined")


equation helpers defined


## C. GF180 device data via pygmid (gm/ID lookup tables)

Same flow as `comp_sizing.ipynb` / `sky130_gff_ip__bgr_1.2v/sizing/sizing_op.ipynb`: locate the `.mat`/`.mat.gz` tables in `../../../Sizing/`, load NMOS/PMOS 3.3 V tables with `pygmid.Lookup`, then for each device pick a $g_m/I_D$ regime and channel length and look the operating point up directly (`lookupVGS`, `lookup('ID_W'|'VT'|'GM'|'STH', ...)`).

* **M1/M2 (input pair):** moderate-to-high $g_m/I_D$ (weak/moderate inversion) — best $g_m$ per µA, lowest input-referred noise for a given current.
* **M3/M4 (mirror load):** similar regime to M1/M2 (their $g_{m3}/g_{m1}$ ratio sets the noise-folding term in B.2).
* **M6 (second-stage driver):** its $g_m$ is *pinned* by the A.2 stability target, so its $g_m/I_D$ is chosen for a **speed/current trade-off** — here a moderate value so $I_7$ stays affordable within the power budget.
* **M5/M7 (tail/bias current sources):** sized last, purely from the current they must carry and a chosen (longer) $L$ for good output resistance / low flicker.

**Sandbox note:** this container does not have the project's `Sizing/` GF180 `.mat` tables mounted (they live in the `nanovolt-chipathon2026` repo, same as `comp_sizing.ipynb` uses). The lookup cell below tries the real tables first — drop this notebook into the project and it will pick them up automatically, exactly like the comparator notebook. If the tables aren't found (as in this sandbox run), it falls back to a clearly-labeled **synthetic long-channel placeholder model** so the rest of the notebook (Section D) still executes end-to-end for structure/logic verification — **replace with a real run before trusting any W/L numbers.**


In [7]:

# --- locate Sizing/ and load the pygmid tables (decompress .mat.gz on the fly) ---
CANDS = ['../../../Sizing',
         '/foss/designs/nanovolt-chipathon2026/designs/Sizing',
         str(Path.home()/'eda/designs/nanovolt-chipathon2026/designs/Sizing')]
SIZING = next((Path(c) for c in CANDS
               if (Path(c)/'gf180mcu_nmos_3v3.mat').exists()
               or (Path(c)/'gf180mcu_nmos_3v3.mat.gz').exists()), None)

USING_REAL_TABLES = SIZING is not None

def scal(x):
    return float(np.asarray(x).ravel()[0])

if USING_REAL_TABLES:
    CACHE = Path(tempfile.gettempdir())/'gmid_gf180'; CACHE.mkdir(exist_ok=True)
    def ensure_mat(stem):
        src = SIZING/f'{stem}.mat'
        if src.exists():
            return str(src)
        dst = CACHE/f'{stem}.mat'
        if not dst.exists():
            with gzip.open(SIZING/f'{stem}.mat.gz','rb') as fi, open(dst,'wb') as fo:
                shutil.copyfileobj(fi, fo)
        return str(dst)
    n = lk(ensure_mat('gf180mcu_nmos_3v3'))
    p = lk(ensure_mat('gf180mcu_pmos_3v3'))
    print("loaded REAL GF180 nmos/pmos gm/ID tables from", SIZING)
else:
    print("[warn] Sizing/ GF180 .mat tables not found in this sandbox.")
    print("       Falling back to a SYNTHETIC placeholder gm/ID model so the notebook")
    print("       still runs end-to-end. Re-run inside the project repo for real numbers.")

    class SyntheticLookup:
        # Very rough square-law/long-channel stand-in for pygmid.Lookup, ONLY for
        # letting this notebook execute without the real GF180 characterization data.
        # Not a substitute for the real tables -- gm/ID vs Vov mapping, VT, and ID/W
        # are all approximate placeholders.
        def __init__(self, kind='n'):
            self.VTH0 = 0.45 if kind == 'n' else -0.45
            self.KP   = 180e-6 if kind == 'n' else 60e-6   # A/V^2, rough GF180 180nm-ish placeholder
        def lookupVGS(self, GM_ID, L, VDS, VSB=0.0):
            # gm/ID = 2/Vov for square law -> Vov = 2/(gm/ID)
            Vov = 2.0/GM_ID
            vgs = abs(self.VTH0) + Vov
            return vgs if self.VTH0 >= 0 else -vgs
        def lookup(self, key, L, VGS, VDS, VSB=0.0):
            Vov = abs(VGS) - abs(self.VTH0)
            Vov = max(Vov, 1e-3)
            if key == 'VT':
                return self.VTH0
            if key == 'ID_W':
                # Id/W = 0.5*KP/L*Vov^2 (square law, L in um -> KP already area-normalized here, rough)
                return 0.5*self.KP/(L*1e-6)*Vov**2 * 1e-6   # crude scaling to stay in a sane A/um range
            if key == 'GM':
                idw = self.lookup('ID_W', L, VGS, VDS, VSB)
                gm_id = 2.0/Vov
                return gm_id*idw*10.0   # placeholder width-normalized gm; scaled in Section D via width anyway
            if key == 'STH':
                gm = self.lookup('GM', L, VGS, VDS, VSB)
                return 4*kT*(2/3)*gm   # assume gamma=2/3 exactly in the synthetic model
            raise KeyError(key)
    n = SyntheticLookup('n')
    p = SyntheticLookup('p')


loaded REAL GF180 nmos/pmos gm/ID tables from ../../../Sizing


### C.1 Pick gm/ID regimes and read GF180 operating points


In [8]:

# input pair M1/M2 : moderate-high gm/ID (good gm-per-uA, low noise), biased near VDDA/2
gmid_in, L_in = 14, 0.5
vgs_in = scal(n.lookupVGS(GM_ID=gmid_in, L=L_in, VDS=VDDA/2, VSB=0.0))
vth_in = scal(n.lookup('VT',   L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
idw_in = scal(n.lookup('ID_W', L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))   # A/um
vov_in = vgs_in - vth_in

# mirror load M3/M4 (PMOS) : similar regime, sets the noise-folding gm3/gm1 term
gmid_m34, L_m34 = 12, 0.5
vgs_m34 = scal(p.lookupVGS(GM_ID=gmid_m34, L=L_m34, VDS=VDDA/2, VSB=0.0))
vth_m34 = scal(p.lookup('VT',   L=L_m34, VGS=vgs_m34, VDS=VDDA/2, VSB=0.0))
idw_m34 = scal(p.lookup('ID_W', L=L_m34, VGS=vgs_m34, VDS=VDDA/2, VSB=0.0))

# stage-2 driver M6 (NMOS common source) : gm is PINNED by A.2, gm/ID chosen for current efficiency
gmid_o6, L_o6 = 10, 0.28
vgs_o6 = scal(n.lookupVGS(GM_ID=gmid_o6, L=L_o6, VDS=VDDA/2, VSB=0.0))
vth_o6 = scal(n.lookup('VT',   L=L_o6, VGS=vgs_o6, VDS=VDDA/2, VSB=0.0))
idw_o6 = scal(n.lookup('ID_W', L=L_o6, VGS=vgs_o6, VDS=VDDA/2, VSB=0.0))
vov_o6 = vgs_o6 - vth_o6

# stage-2 load M7 (PMOS current source)
gmid_o7, L_o7 = 8, 0.5
vgs_o7 = scal(p.lookupVGS(GM_ID=gmid_o7, L=L_o7, VDS=VDDA/2, VSB=0.0))
vth_o7 = scal(p.lookup('VT',   L=L_o7, VGS=vgs_o7, VDS=VDDA/2, VSB=0.0))
idw_o7 = scal(p.lookup('ID_W', L=L_o7, VGS=vgs_o7, VDS=VDDA/2, VSB=0.0))
vov_o7 = vgs_o7 - vth_o7

print(f"M1/M2  in : gm/Id={gmid_in} L={L_in}um  Vgs={vgs_in:.3f} Vth={vth_in:.3f} Vov={vov_in:+.3f}  Id/W={idw_in*1e6:.3f} uA/um")
print(f"M3/M4 mir : gm/Id={gmid_m34} L={L_m34}um Vgs={vgs_m34:.3f} Vth={vth_m34:.3f}             Id/W={idw_m34*1e6:.3f} uA/um")
print(f"M6    o/p : gm/Id={gmid_o6} L={L_o6}um  Vgs={vgs_o6:.3f} Vth={vth_o6:.3f} Vov={vov_o6:+.3f}  Id/W={idw_o6*1e6:.3f} uA/um")
print(f"M7   load : gm/Id={gmid_o7} L={L_o7}um  Vgs={vgs_o7:.3f} Vth={vth_o7:.3f} Vov={vov_o7:+.3f}  Id/W={idw_o7*1e6:.3f} uA/um")

# --- thermal-noise factor gamma from the noise-complete table (STH), same as comp_sizing.ipynb ---
gamma_fallback = 2/3
gm_in  = scal(n.lookup('GM',  L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
try:
    sth_in = scal(n.lookup('STH', L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
    gamma_tab = sth_in/(4*kT*gm_in)
    gamma = gamma_tab if 0.3 < gamma_tab < 2.0 else gamma_fallback
    print(f"\ngamma (from STH) = {gamma:.3f}")
except Exception as e:
    gamma = gamma_fallback
    print(f"\n[warn] STH lookup unavailable ({e}); using fallback gamma={gamma:.3f}")


M1/M2  in : gm/Id=14 L=0.5um  Vgs=0.762 Vth=0.710 Vov=+0.052  Id/W=1.755 uA/um
M3/M4 mir : gm/Id=12 L=0.5um Vgs=0.900 Vth=0.786             Id/W=0.788 uA/um
M6    o/p : gm/Id=10 L=0.28um  Vgs=0.733 Vth=0.624 Vov=+0.110  Id/W=6.958 uA/um
M7   load : gm/Id=8 L=0.5um  Vgs=0.997 Vth=0.786 Vov=+0.211  Id/W=2.051 uA/um

gamma (from STH) = 0.713


## D. Equations → sizing numbers

Feed the GF180 operating points into the boxed A/B equations, following the constraint order from Section A.4: $g_{m6}$ is pinned first (stability), which fixes $I_7$; whatever remains of the power budget goes to $I_5$ (stage 1), which fixes $g_{m1}$, $C_c$, and everything downstream.


In [9]:

# D.1  stability target -> gm6 -> I7 (Section A.2 + C.1 gm/Id choice)
gm6 = gm6_target
I7  = gm6/gmid_o6
print(f"D.1 STABILITY: gm6_target={gm6*1e6:.1f} uS  ->  I7 = gm6/gmid_o6 = {I7*1e6:.2f} uA")

# D.2  remaining power budget -> I5 -> gm1 -> Cc (Section A.1, A.4)
I_total = P_ota_budget/VDDA
I5 = I_total - I7
assert I5 > 0, "Power budget too tight for the target GBW/PM/CL -- relax GBW, k_pm, or CL, or raise P_ota_budget"
gm1 = gmid_in*(I5/2)                 # each of M1,M2 carries I5/2
Cc  = cc_from_gbw(gm1, GBW)
GBW_check = gm1/(2*np.pi*Cc)
print(f"D.2 POWER    : I_total={I_total*1e6:.2f}uA -> I5={I5*1e6:.2f}uA (I7 already took {I7*1e6:.2f}uA)")
print(f"               gm1={gm1*1e6:.1f}uS -> Cc={Cc*1e12:.3f}pF  (Cc/CL={Cc/CL:.2f}, cf. classic Cc>=0.22*CL rule)")
print(f"               GBW check = {GBW_check/1e6:.2f} MHz (should equal design target {GBW/1e6:.1f} MHz)")

# D.3  nulling resistor for pole-zero cancellation (Section B.1)
Rz = rz_zero_cancel(Cc, CL, gm6)
print(f"D.3 COMP     : Rz (zero on top of -p2) = {Rz/1e3:.2f} kOhm   (1/gm6 = {1/gm6/1e3:.2f} kOhm, for reference)")

# D.4  slew rate check (Section A.3)
SR = slew_rate(I5, Cc)
sr_margin_db = 20*np.log10(SR/SR_req)
print(f"D.4 SLEW     : SR = I5/Cc = {SR/1e6:.2f} V/us  vs required {SR_req/1e6:.2f} V/us "
      f"-> {'OK' if SR>=SR_req else 'FAIL'} ({sr_margin_db:+.1f} dB margin)")

# D.5  output swing check
Vout_min = vov_o6
Vout_max = VDDA - vov_o7
swing_avail_pp = Vout_max - Vout_min
print(f"D.5 SWING    : available single-ended swing ~ VDDA-Vov6-Vov7 = {swing_avail_pp:.2f} Vpp "
      f"vs required {Vswing_se_pp:.2f} Vpp -> {'OK, large margin' if swing_avail_pp>=Vswing_se_pp else 'FAIL'}")

# D.6  input-referred noise (informational -- no explicit noise spec was given for the PGA)
gm3 = gmid_m34*(I5/2)
psd_in = input_ref_noise_psd(gm1, gm3, gamma)
noise_bw = (np.pi/2)*GBW      # single-pole brick-wall-equivalent noise bandwidth
vn_rms = np.sqrt(integrated_noise(psd_in, noise_bw))
print(f"D.6 NOISE (info): Sv_in = {psd_in*1e18:.2f} nV^2/Hz  ({np.sqrt(psd_in)*1e9:.2f} nV/rtHz), "
      f"integrated over {noise_bw/1e6:.1f}MHz noise BW -> vn_rms ~ {vn_rms*1e6:.1f} uVrms (input-referred)")

# D.7  DC gain estimate (needs an ro/gds figure; try the table, else assume a typical intrinsic gain)
try:
    gds1 = scal(n.lookup('GDS', L=L_in, VGS=vgs_in, VDS=VDDA/2, VSB=0.0))
    gds3 = scal(p.lookup('GDS', L=L_m34, VGS=vgs_m34, VDS=VDDA/2, VSB=0.0))
    Ro1 = 1/(gds1+gds3)
    gds6 = scal(n.lookup('GDS', L=L_o6, VGS=vgs_o6, VDS=VDDA/2, VSB=0.0))
    gds7 = scal(p.lookup('GDS', L=L_o7, VGS=vgs_o7, VDS=VDDA/2, VSB=0.0))
    Ro2 = 1/(gds6+gds7)
    Av = dc_gain(gm1, Ro1, gm6, Ro2)
    print(f"D.7 DC GAIN  : Ro1={Ro1/1e3:.1f}kOhm Ro2={Ro2/1e3:.1f}kOhm -> Av = {Av:.0f} V/V ({20*np.log10(Av):.1f} dB)")
except Exception as e:
    intrinsic_gain_assumed = 25.0   # typical gm*ro for a short-channel GF180 device, placeholder
    Av = gm1*intrinsic_gain_assumed*gm6*intrinsic_gain_assumed / max(gm1,1e-12) / max(gm6,1e-12) * (intrinsic_gain_assumed**0)  # keep simple:
    Av = intrinsic_gain_assumed**2
    print(f"D.7 DC GAIN  : GDS lookup unavailable ({e}); assuming intrinsic gain gm*ro~{intrinsic_gain_assumed:.0f} per stage "
          f"-> Av ~ {Av:.0f} V/V ({20*np.log10(Av):.1f} dB) [placeholder]")


D.1 STABILITY: gm6_target=312.5 uS  ->  I7 = gm6/gmid_o6 = 31.25 uA
D.2 POWER    : I_total=60.61uA -> I5=29.36uA (I7 already took 31.25uA)
               gm1=205.5uS -> Cc=0.654pF  (Cc/CL=0.26, cf. classic Cc>=0.22*CL rule)
               GBW check = 50.00 MHz (should equal design target 50.0 MHz)
D.3 COMP     : Rz (zero on top of -p2) = 15.43 kOhm   (1/gm6 = 3.20 kOhm, for reference)
D.4 SLEW     : SR = I5/Cc = 44.88 V/us  vs required 12.57 V/us -> OK (+11.1 dB margin)
D.5 SWING    : available single-ended swing ~ VDDA-Vov6-Vov7 = 2.98 Vpp vs required 1.00 Vpp -> OK, large margin
D.6 NOISE (info): Sv_in = 199.64 nV^2/Hz  (14.13 nV/rtHz), integrated over 78.5MHz noise BW -> vn_rms ~ 125.2 uVrms (input-referred)
D.7 DC GAIN  : Ro1=1191.3kOhm Ro2=84.7kOhm -> Av = 6482 V/V (76.2 dB)


In [10]:

snap = lambda w: max(round(w*2)/2, 0.5)   # round to 0.5 um grid, >= 0.5 um

W_in  = (I5/2)/idw_in
W_m34 = (I5/2)/idw_m34
W_o6  = I7/idw_o6
W_o7  = I7/idw_o7
L_tail = 1.0                              # longer L for the tail devices -> better output resistance / lower flicker
idw_tail_n = idw_in                       # reuse the input-pair NMOS density as a rough estimate for M5's regime
idw_tail_p = idw_o7
W_tail5 = I5/idw_tail_n
W_tail7b = 0.0  # M7 IS the stage-2 tail/load, already sized above

sizing = pd.DataFrame([
    dict(Device='M1,M2 input pair',   W_um=snap(W_in),  L_um=L_in,  nf=2, gm_id=gmid_in,  I_branch_uA=I5/2*1e6, set_by='gm1 -> GBW (D.2)'),
    dict(Device='M3,M4 mirror load',  W_um=snap(W_m34), L_um=L_m34, nf=2, gm_id=gmid_m34, I_branch_uA=I5/2*1e6, set_by='noise gm3/gm1 term (B.2)'),
    dict(Device='M5 tail (stage 1)',  W_um=snap(W_tail5),L_um=L_tail,nf=1, gm_id='-',      I_branch_uA=I5*1e6,   set_by='sets I5 (power budget, D.2)'),
    dict(Device='M6 stage-2 driver',  W_um=snap(W_o6),   L_um=L_o6,  nf=2, gm_id=gmid_o6,  I_branch_uA=I7*1e6,   set_by='gm6 -> phase margin (D.1)'),
    dict(Device='M7 stage-2 load',    W_um=snap(W_o7),   L_um=L_o7,  nf=2, gm_id=gmid_o7,  I_branch_uA=I7*1e6,   set_by='mirrors I7'),
])
print(sizing.to_string(index=False))

print(f"\nSUMMARY: Cc={Cc*1e12:.2f}pF  Rz={Rz/1e3:.2f}kOhm  GBW={GBW_check/1e6:.1f}MHz  "
      f"SR={SR/1e6:.1f}V/us  Av~{20*np.log10(Av):.0f}dB  vn_rms~{vn_rms*1e6:.1f}uVrms")
print(f"Per-OTA current: I5={I5*1e6:.1f}uA + I7={I7*1e6:.1f}uA = {(I5+I7)*1e6:.1f}uA "
      f"-> P_ota={(I5+I7)*VDDA*1e6:.1f}uW  (budget {P_ota_budget*1e6:.0f}uW)")
print(f"Whole PGA ({n_ota} OTAs, core only, excl. CMFB/bias): "
      f"P_core~{(I5+I7)*VDDA*n_ota*1e6:.1f}uW  (spec <{P_total_budget*1e6:.0f}uW)")

if not USING_REAL_TABLES:
    print("\n[!] Numbers above were generated with the SYNTHETIC placeholder gm/ID model")
    print("    (Section C) because the real GF180 Sizing/ tables were not found in this")
    print("    sandbox. Re-run this notebook inside the project repo before trusting any")
    print("    W/L, Cc, Rz or power figure.")


           Device  W_um  L_um  nf gm_id  I_branch_uA                      set_by
 M1,M2 input pair   8.5  0.50   2    14    14.678030            gm1 -> GBW (D.2)
M3,M4 mirror load  18.5  0.50   2    12    14.678030    noise gm3/gm1 term (B.2)
M5 tail (stage 1)  16.5  1.00   1     -    29.356061 sets I5 (power budget, D.2)
M6 stage-2 driver   4.5  0.28   2    10    31.250000   gm6 -> phase margin (D.1)
  M7 stage-2 load  15.0  0.50   2     8    31.250000                  mirrors I7

SUMMARY: Cc=0.65pF  Rz=15.43kOhm  GBW=50.0MHz  SR=44.9V/us  Av~76dB  vn_rms~125.2uVrms
Per-OTA current: I5=29.4uA + I7=31.2uA = 60.6uA -> P_ota=200.0uW  (budget 200uW)
Whole PGA (2 OTAs, core only, excl. CMFB/bias): P_core~400.0uW  (spec <400uW)


## F. Common-mode feedback (CMFB) sizing

Matches the CMFB block topology drawn alongside the two-stage op-amp (resistive
common-mode sensing + differential error amplifier + current mirror feeding
back into the output stage bias), reusing the same device names as the
schematic: **R2, R3** (sensing divider), **M9** (CMFB tail), **M10, M11**
(sensing differential pair, gates = sensed $V_{CM}$ and reference $V_{CM,ref}$),
**M12, M13** (PMOS mirror load that returns the correction current into the
stage-2 bias node, i.e. the gates of M7/M8 in the core OTA).

### F.1 Common-mode sensing network (R2, R3)
$$V_{CM,\mathrm{sense}}=\frac{R_3}{R_2+R_3}V_\mathrm{out-}+\frac{R_2}{R_2+R_3}V_\mathrm{out+}
\;\xrightarrow{R_2=R_3=R_{cm}}\;\boxed{\,V_{CM,\mathrm{sense}}=\dfrac{V_\mathrm{out+}+V_\mathrm{out-}}{2}\,}.$$
Two competing constraints size $R_{cm}$:
* **Loading:** the divider loads the OTA's own high-impedance output node, so $R_{cm}\gg R_{o2}$ (the stage-2 output resistance from Section D.7) to avoid eating differential-mode DC gain — rule of thumb $R_{cm}\ge10\,R_{o2}$.
* **Sensing-node pole:** the divider's Thevenin resistance at the sense node, $R_{th}=R_2\|R_3=R_{cm}/2$, forms a pole with the gate capacitance of M10, $f_{sense}=1/(2\pi R_{th}C_{g10})$, which must stay well above the CMFB loop's own unity-gain frequency (F.4) or it eats the CMFB loop's phase margin.

### F.2 Error amplifier tail & pair (M9, M10, M11)
Same gm/ID methodology as the core OTA: budget a **CMFB current overhead** on top of the core $I_5+I_7$ (Section D.2), split into the M9 tail, then $g_{m10}=g_{m11}=(g_m/I_D)_\mathrm{cmfb}\cdot(I_9/2)$.

### F.3 Mirror-back devices (M12, M13)
Sized to carry $I_9/2$ each; a mirror ratio $k_{mirror}=I_{12,13}/I_{7}$ (device-width ratio at matched $V_{GS}$) sets how much correction current the CMFB injects into the stage-2 bias node per volt of common-mode error.

### F.4 CMFB loop gain & bandwidth (first-order check)
Treating the CMFB path the same way as the core differential loop's DC-gain estimate (Section D.7):
$$\boxed{\,A_{cmfb}\approx g_{m10}\,R_{o2}\,},\qquad \boxed{\,\mathrm{GBW}_{cmfb}\approx\dfrac{g_{m10}}{2\pi\,C_{node}}\,},$$
where $C_{node}$ is the dominant capacitance the M10/M11 drain (mirror input) node presents — approximated here by the M12/M13 gate-drain capacitance loading, which this hand-analysis does not have from `pygmid` alone, so a **fixed multiple of $C_c$** is used as a placeholder ($C_{node}\approx C_c$, i.e. assume the CMFB's own compensation node behaves similarly to the core loop's Miller node). Design guideline (Allen & Holberg, CMFB chapter): the CMFB loop should be **at least as fast as, and ideally ~2× faster than**, the differential-mode loop, so it settles common-mode transients without becoming the system's bottleneck or fighting the main loop:
$$\boxed{\,\mathrm{GBW}_{cmfb}\ge 2\times \mathrm{GBW}\,}.$$
This is a rule-of-thumb sizing pass only — **confirm with a dedicated common-mode AC loop-gain simulation** (Section G).


In [11]:

# F.1 sensing network
Vcm_ref = VDDA/2   # target output common-mode

# Ro2 may or may not exist as a real number depending on whether Section D.7 found a
# real GDS lookup (NOTE: 'Ro2' is also used as a SymPy Symbol name in Section B, so we
# must check the TYPE here, not just presence, or we'd pick up that leftover symbol).
if 'Ro2' not in dir() or not isinstance(Ro2, (int, float, np.floating)):
    Ro2 = intrinsic_gain_assumed/gm6      # fallback single-device ro estimate, mirrors D.7's placeholder path
    print(f"[info] Ro2 not available as a numeric value from D.7 GDS lookup; using placeholder Ro2 = intrinsic_gain/gm6 = {Ro2/1e3:.1f} kOhm")

Rcm_min_loading = 10*Ro2
Rcm = Rcm_min_loading         # design choice: sit right at the 10x-Ro2 loading floor to keep area down
R2 = R3 = Rcm

Cg10_est = 5e-15              # F, rough gate-cap estimate for a small sensing-pair device (refine once M10 is sized)
Rth_sense = Rcm/2
f_sense = 1/(2*np.pi*Rth_sense*Cg10_est)

print(f"F.1 SENSE   : Vcm_ref={Vcm_ref:.3f} V,  R2=R3={Rcm/1e3:.1f} kOhm (>= 10*Ro2={Rcm_min_loading/1e3:.1f} kOhm)")
print(f"              sensing-node pole f_sense ~ {f_sense/1e6:.1f} MHz (Cg10 estimate={Cg10_est*1e15:.1f}fF) -- check vs F.4 GBW_cmfb")


F.1 SENSE   : Vcm_ref=1.650 V,  R2=R3=847.3 kOhm (>= 10*Ro2=847.3 kOhm)
              sensing-node pole f_sense ~ 75.1 MHz (Cg10 estimate=5.0fF) -- check vs F.4 GBW_cmfb


In [12]:

# F.2 CMFB current budget, tail (M9), and sensing pair (M10,M11)
cmfb_overhead_frac = 0.15                       # 15% of core OTA current, per Section G caveat in the original notebook
I_core = I5 + I7
I9 = cmfb_overhead_frac*I_core                  # CMFB tail current
print(f"F.2 BUDGET  : I_core(core OTA)={I_core*1e6:.2f}uA  ->  I9 (CMFB tail) = {cmfb_overhead_frac*100:.0f}% = {I9*1e6:.2f}uA")

gmid_cmfb, L_cmfb = 12, 0.5
vgs_cmfb = scal(n.lookupVGS(GM_ID=gmid_cmfb, L=L_cmfb, VDS=VDDA/2, VSB=0.0))
vth_cmfb = scal(n.lookup('VT',   L=L_cmfb, VGS=vgs_cmfb, VDS=VDDA/2, VSB=0.0))
idw_cmfb = scal(n.lookup('ID_W', L=L_cmfb, VGS=vgs_cmfb, VDS=VDDA/2, VSB=0.0))
vov_cmfb = vgs_cmfb - vth_cmfb

gm10 = gmid_cmfb*(I9/2)
W_1011 = (I9/2)/idw_cmfb
print(f"      M10/M11: gm/Id={gmid_cmfb} L={L_cmfb}um Vgs={vgs_cmfb:.3f} Vov={vov_cmfb:+.3f} Id/W={idw_cmfb*1e6:.3f}uA/um "
      f"-> gm10={gm10*1e6:.1f}uS, W={W_1011:.2f}um (I_branch={I9/2*1e6:.2f}uA)")

# tail M9 (NMOS, longer L for good current-source output resistance, same regime as core tail M5)
L_tail_cmfb = 1.0
W_tail9 = I9/idw_in     # reuse input-pair NMOS current density as the tail-device estimate (same as M5 in Section D)
print(f"      M9 tail : L={L_tail_cmfb}um  W={W_tail9:.2f}um  (I_branch={I9*1e6:.2f}uA)")


F.2 BUDGET  : I_core(core OTA)=60.61uA  ->  I9 (CMFB tail) = 15% = 9.09uA
      M10/M11: gm/Id=12 L=0.5um Vgs=0.798 Vov=+0.088 Id/W=2.730uA/um -> gm10=54.5uS, W=1.66um (I_branch=4.55uA)
      M9 tail : L=1.0um  W=5.18um  (I_branch=9.09uA)


In [13]:

# F.3 mirror-back devices M12, M13 (PMOS), matched to the I7/M7 current density, unity mirror ratio assumed
k_mirror = 1.0
I_1213 = I9/2                       # each carries half the CMFB tail current (mirrors M10/M11 branch currents)
W_1213 = I_1213/idw_o7              # reuse stage-2 PMOS load current density (M7's regime) for consistency
print(f"F.3 MIRROR  : k_mirror={k_mirror:.1f} (M12,M13 sized in M7's gm/Id regime for a matched bias node)")
print(f"      M12/M13: L={L_o7}um  W={W_1213:.2f}um  (I_branch={I_1213*1e6:.2f}uA)")


F.3 MIRROR  : k_mirror=1.0 (M12,M13 sized in M7's gm/Id regime for a matched bias node)
      M12/M13: L=0.5um  W=2.22um  (I_branch=4.55uA)


In [14]:

# F.4 loop gain / bandwidth sanity check
A_cmfb = gm10*Ro2
Cnode_cmfb = Cc                      # placeholder: assume similar order of magnitude to the core Miller node
GBW_cmfb = gm10/(2*np.pi*Cnode_cmfb)

print(f"F.4 LOOP    : A_cmfb ~ gm10*Ro2 = {A_cmfb:.1f} V/V ({20*np.log10(A_cmfb):.1f} dB)")
print(f"              GBW_cmfb ~ gm10/(2*pi*Cnode) = {GBW_cmfb/1e6:.1f} MHz  vs  2x core GBW = {2*GBW/1e6:.1f} MHz "
      f"-> {'OK' if GBW_cmfb>=2*GBW else 'MARGINAL/FAIL -- raise I9 or gmid_cmfb, or confirm Cnode_cmfb with real parasitics'}")
print(f"              sensing-node pole f_sense={f_sense/1e6:.1f} MHz vs GBW_cmfb={GBW_cmfb/1e6:.1f} MHz "
      f"-> {'OK, pole is well above loop unity gain' if f_sense>=3*GBW_cmfb else 'CHECK -- sensing pole may eat CMFB phase margin'}")


F.4 LOOP    : A_cmfb ~ gm10*Ro2 = 4.6 V/V (13.3 dB)
              GBW_cmfb ~ gm10/(2*pi*Cnode) = 13.3 MHz  vs  2x core GBW = 100.0 MHz -> MARGINAL/FAIL -- raise I9 or gmid_cmfb, or confirm Cnode_cmfb with real parasitics
              sensing-node pole f_sense=75.1 MHz vs GBW_cmfb=13.3 MHz -> OK, pole is well above loop unity gain


In [15]:

# F.5 updated sizing table (core OTA + CMFB) and updated total power
cmfb_rows = pd.DataFrame([
    dict(Device='R2, R3 (sense)',     W_um='-',        L_um='-',   nf='-', gm_id='-',      I_branch_uA=0.0,          set_by=f'Rcm={Rcm/1e3:.0f}kOhm, 10x Ro2 loading floor (F.1)'),
    dict(Device='M9 CMFB tail',       W_um=snap(W_tail9), L_um=L_tail_cmfb, nf=1, gm_id='-', I_branch_uA=I9*1e6,      set_by='15% overhead on I_core (F.2)'),
    dict(Device='M10,M11 sense pair', W_um=snap(W_1011),  L_um=L_cmfb,     nf=2, gm_id=gmid_cmfb, I_branch_uA=I9/2*1e6, set_by='gm10 -> A_cmfb, GBW_cmfb (F.2/F.4)'),
    dict(Device='M12,M13 mirror',     W_um=snap(W_1213),  L_um=L_o7,       nf=2, gm_id='-', I_branch_uA=I_1213*1e6,   set_by=f'k_mirror={k_mirror:.1f}, feeds back into M7/M8 bias (F.3)'),
])

sizing_full = pd.concat([sizing, cmfb_rows], ignore_index=True)
print(sizing_full.to_string(index=False))

I_total_with_cmfb = I_core + I9
P_ota_with_cmfb   = I_total_with_cmfb*VDDA
P_pga_with_cmfb   = P_ota_with_cmfb*n_ota

print(f"\nPer-OTA current incl. CMFB: {I_total_with_cmfb*1e6:.2f} uA -> P_ota = {P_ota_with_cmfb*1e6:.1f} uW "
      f"(core-only budget was {P_ota_budget*1e6:.0f} uW)")
print(f"Whole PGA ({n_ota} OTAs) incl. CMFB: P_total ~ {P_pga_with_cmfb*1e6:.1f} uW  vs spec <{P_total_budget*1e6:.0f} uW "
      f"-> {'OK' if P_pga_with_cmfb<P_total_budget else 'OVER BUDGET -- tighten cmfb_overhead_frac, gmid choices, or GBW/k_pm margins'}")


            Device  W_um  L_um nf gm_id  I_branch_uA                                         set_by
  M1,M2 input pair   8.5   0.5  2    14    14.678030                               gm1 -> GBW (D.2)
 M3,M4 mirror load  18.5   0.5  2    12    14.678030                       noise gm3/gm1 term (B.2)
 M5 tail (stage 1)  16.5   1.0  1     -    29.356061                    sets I5 (power budget, D.2)
 M6 stage-2 driver   4.5  0.28  2    10    31.250000                      gm6 -> phase margin (D.1)
   M7 stage-2 load  15.0   0.5  2     8    31.250000                                     mirrors I7
    R2, R3 (sense)     -     -  -     -     0.000000       Rcm=847kOhm, 10x Ro2 loading floor (F.1)
      M9 CMFB tail   5.0   1.0  1     -     9.090909                   15% overhead on I_core (F.2)
M10,M11 sense pair   1.5   0.5  2    12     4.545455             gm10 -> A_cmfb, GBW_cmfb (F.2/F.4)
    M12,M13 mirror   2.0   0.5  2     -     4.545455 k_mirror=1.0, feeds back into M7/M8 bias (F.3)


## H. MIM capacitor sizing (`Cc` and the biquad integrator cap `Cint`)

GF180MCU offers three MIM-capacitor densities, but **only one may be used per
maskset/process flow** — it's a single global choice for the whole die, not a
per-instance option. The table below (from the PDK electrical spec sheet) is
built into `mim_density_fF_um2` below as a **single switch**: change it once
and every capacitor in this section resizes consistently.

| Density | Vc1 (voltage coeff., typ) | Max operating voltage | BVox (typ) | Matching (1 pF, adjacent) |
|---|---|---|---|---|
| 1.0 fF/µm² | ~+6 ppm/V | 20 V | 40 V | ~1% |
| 1.5 fF/µm² | ~+7.6 ppm/V | 10 V | 30 V | ~1% |
| 2.0 fF/µm² | ~−28.8 ppm/V (≈4-5x more nonlinear) | 6.6 V | 24 V | ~1% |

Two checks matter for picking the density and sizing the plates:

1. **Breakdown/operating-voltage margin:** the cap's max operating voltage must clear `VDDA` (3.3 V here) with margin.
2. **Signal-dependent nonlinearity:** $\Delta C/C \approx V_{c1}\cdot V_\mathrm{swing}$ — both `Cc` (Miller cap, sees close to the full output swing since it's tied straight to the output node) and `Cint` (biquad integrator cap, sees the filtered signal swing) sit in the signal path, so this voltage-coefficient term is a direct THD contributor, unlike a pure decoupling cap.


In [16]:

# H.1 -- MIM density options (from the PDK electrical spec sheet, Sec. 6.2)
mim_props = {
    1.0: dict(Vc1_ppm_per_V=6.0,   TC1_ppm_per_K=10.0, BVox_typ_V=40, Vmax_op_V=20.0, matching_pct=1.0),
    1.5: dict(Vc1_ppm_per_V=7.6,   TC1_ppm_per_K=13.3, BVox_typ_V=30, Vmax_op_V=10.0, matching_pct=1.0),
    2.0: dict(Vc1_ppm_per_V=-28.8, TC1_ppm_per_K=18.8, BVox_typ_V=24, Vmax_op_V=6.6,  matching_pct=1.0),
}

mim_density_fF_um2 = 2.0     # <-- single global switch: 1.0 / 1.5 / 2.0 (must match the whole-die maskset choice)
props = mim_props[mim_density_fF_um2]

print(f"Selected MIM density : {mim_density_fF_um2} fF/um^2")
print(f"  Vc1 (voltage coeff): {props['Vc1_ppm_per_V']:+.1f} ppm/V")
margin_ok = props['Vmax_op_V'] > VDDA
margin_txt = f"OK, {props['Vmax_op_V']/VDDA:.2f}x margin" if margin_ok else "FAIL, VDDA exceeds rating!"
print(f"  Max operating volt.: {props['Vmax_op_V']:.1f} V   (VDDA={VDDA} V -> {margin_txt})")
print(f"  Matching (1pF, adj): ~{props['matching_pct']:.1f}%")


Selected MIM density : 2.0 fF/um^2
  Vc1 (voltage coeff): -28.8 ppm/V
  Max operating volt.: 6.6 V   (VDDA=3.3 V -> OK, 2.00x margin)
  Matching (1pF, adj): ~1.0%


In [17]:

# H.2 -- plate area / square-side sizing for Cc (Section D) and Cint (Section 0 config)
def mim_size(C_F, density_fF_um2):
    area_um2 = (C_F*1e15)/density_fF_um2      # fF / (fF/um^2) = um^2
    side_um  = np.sqrt(area_um2)              # assuming a square plate for a first pass
    return area_um2, side_um

area_Cc, side_Cc = mim_size(Cc, mim_density_fF_um2)
area_Cint, side_Cint = mim_size(Cint, mim_density_fF_um2)

mim_table = pd.DataFrame([
    dict(Cap='Cc (compensation)', C_pF=Cc*1e12,   Area_um2=area_Cc,   Side_um=side_Cc,   Role='OTA Miller cap (Section D.2)'),
    dict(Cap='Cint (biquad integrator)', C_pF=Cint*1e12, Area_um2=area_Cint, Side_um=side_Cint, Role='per-integrator cap, from Section 0 config (shared with pga_towthomas_sizing.ipynb)'),
])
print(mim_table.to_string(index=False))
print(f"\n(Both instantiated as {mim_density_fF_um2} fF/um^2 MIM caps, square plates as a first-pass layout estimate;"
      f" real layout will use the PDK's MIM generator/PCell with DRC-legal aspect ratio and fill.)")


                     Cap     C_pF    Area_um2   Side_um                                                                               Role
       Cc (compensation) 0.654103  327.051351 18.084561                                                       OTA Miller cap (Section D.2)
Cint (biquad integrator) 2.000000 1000.000000 31.622777 per-integrator cap, from Section 0 config (shared with pga_towthomas_sizing.ipynb)

(Both instantiated as 2.0 fF/um^2 MIM caps, square plates as a first-pass layout estimate; real layout will use the PDK's MIM generator/PCell with DRC-legal aspect ratio and fill.)


In [18]:

# H.3 -- signal-dependent nonlinearity estimate (Delta C/C ~ Vc1 * V_swing)
V_across_Cc   = Vswing_se_pp     # Cc ties directly to the output node -> assume it sees ~full single-ended swing
V_across_Cint = Vswing_se_pp     # integrator cap also sits in the filtered signal path -> same conservative assumption

dCoverC_Cc   = props['Vc1_ppm_per_V']*1e-6*V_across_Cc
dCoverC_Cint = props['Vc1_ppm_per_V']*1e-6*V_across_Cint

print(f"H.3 NONLINEARITY (Delta C/C ~= Vc1 * V_swing, conservative full single-ended swing assumed):")
print(f"   Cc  : V_across~{V_across_Cc:.2f}Vpp -> Delta C/C ~ {dCoverC_Cc*1e6:+.1f} ppm ({dCoverC_Cc*100:+.4f} %)")
print(f"   Cint: V_across~{V_across_Cint:.2f}Vpp -> Delta C/C ~ {dCoverC_Cint*1e6:+.1f} ppm ({dCoverC_Cint*100:+.4f} %)")
print()
print("   This is a first-order estimate of the MIM cap's OWN contribution to signal-dependent gain")
print("   error / distortion; it does not include transistor-level (gm, ro) nonlinearity, which is")
print("   typically the dominant THD source in this kind of two-stage OTA. Compare against the PGA's")
print("   overall THD/SFDR budget (not stated in the original spec table) before treating this as pass/fail.")


H.3 NONLINEARITY (Delta C/C ~= Vc1 * V_swing, conservative full single-ended swing assumed):
   Cc  : V_across~1.00Vpp -> Delta C/C ~ -28.8 ppm (-0.0029 %)
   Cint: V_across~1.00Vpp -> Delta C/C ~ -28.8 ppm (-0.0029 %)

   This is a first-order estimate of the MIM cap's OWN contribution to signal-dependent gain
   error / distortion; it does not include transistor-level (gm, ro) nonlinearity, which is
   typically the dominant THD source in this kind of two-stage OTA. Compare against the PGA's
   overall THD/SFDR budget (not stated in the original spec table) before treating this as pass/fail.


In [19]:

# H.4 -- side-by-side comparison across all three densities (for the trade-off discussion)
compare_rows = []
for dens, p in mim_props.items():
    area_c, side_c = mim_size(Cc, dens)
    area_i, side_i = mim_size(Cint, dens)
    compare_rows.append(dict(
        Density_fF_um2=dens,
        Vc1_ppm_per_V=p['Vc1_ppm_per_V'],
        Vmax_op_V=p['Vmax_op_V'],
        VDDA_margin_x=p['Vmax_op_V']/VDDA,
        Cc_area_um2=area_c,
        Cint_area_um2=area_i,
        Total_area_um2=area_c+area_i,
    ))
compare = pd.DataFrame(compare_rows)
print(compare.to_string(index=False))
print(f"\nSelected density = {mim_density_fF_um2} fF/um^2 (change H.1's mim_density_fF_um2 to compare trade-offs directly).")


 Density_fF_um2  Vc1_ppm_per_V  Vmax_op_V  VDDA_margin_x  Cc_area_um2  Cint_area_um2  Total_area_um2
            1.0            6.0       20.0       6.060606   654.102702    2000.000000     2654.102702
            1.5            7.6       10.0       3.030303   436.068468    1333.333333     1769.401801
            2.0          -28.8        6.6       2.000000   327.051351    1000.000000     1327.051351

Selected density = 2.0 fF/um^2 (change H.1's mim_density_fF_um2 to compare trade-offs directly).
